# K-NN Classifier

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.datasets import make_classificationfrom sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("datasets/airlines_delay.csv", sep=",")AirlineUnique = df.Airline.unique()AirportFromUnique = df.AirportFrom.unique()AirportToUnique = df.AirportTo.unique()Airlinelst = list(range(len(AirlineUnique)))df['NumAirline'] = df['Airline']df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)AirportFromlst = list(range(len(AirportFromUnique)))df['NumAirportFrom'] = df['AirportFrom']df['NumAirportFrom'].replace(AirportFromUnique, AirportFromlst, inplace=True)AirportTolst = list(range(len(AirportToUnique)))df['NumAirportTo'] = df['AirportTo']df['NumAirportTo'].replace(AirportToUnique, AirportTolst, inplace=True)df = df.sample(n=10000)X = df[["Length","NumAirline","NumAirportFrom","NumAirportTo","DayOfWeek"]]y = df['Class']

In [ ]:
X_simulated_small, y_simulated_small = make_classification(n_samples=300, n_features=6, n_classes=2, random_state=1)X_simulated_large, y_simulated_large = make_classification(n_samples=15000, n_features=6, n_classes=2, random_state=1)

In [ ]:
scaler = StandardScaler()X_std = scaler.fit_transform(X)X_std = X_stdy = y.values

In [ ]:
def test_accuracy(true, pred):    return sum(a==b for a,b in zip(true,pred))/len(true)

In [ ]:
import matplotlib.pyplot as pltfrom sklearn.neighbors import KNeighborsClassifierX_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.2, random_state=1, shuffle=True)score = []for k in range(1,31):    knn = KNeighborsClassifier(n_neighbors=k)    knn.fit(X_train,y_train)    score.append(knn.score(X_test,y_test))plt.plot(range(1,31), score)plt.xlabel('K')plt.ylabel('Accuracy')plt.show()

In [ ]:
from math import sqrtdef euclidean(row1, row2):    return sqrt(np.sum((row1 - row2)**2))

In [ ]:
from joblib import Parallel, delayedimport multiprocessingstart=time.time()acc=[]for i in range(10):    X_train, X_test, y_train, y_test = train_test_split(X_simulated_small,y_simulated_small,test_size=0.2,random_state=i)    num_cores = multiprocessing.cpu_count()    inputs = range(len(X_test))    def multi_run(t):        return predict_classification(X_train,y_train,X_test[t],25)    results = Parallel(n_jobs=num_cores)(delayed(multi_run)(i) for i in inputs)    acc.append(test_accuracy(y_test, results))print('Prediction accuracy of model:', sum(acc)/len(acc))print('Training time for KNN:', time.time()-start)

In [ ]:
start=time.time()acc=[]for i in range(1):    X_train, X_test, y_train, y_test = train_test_split(X_simulated_large,y_simulated_large,test_size=0.2,random_state=i)    num_cores = multiprocessing.cpu_count()    inputs = range(len(X_test))    def multi_run(t):        return predict_classification(X_train,y_train,X_test[t],25)    results = Parallel(n_jobs=num_cores)(delayed(multi_run)(i) for i in inputs)    acc.append(test_accuracy(y_test, results))print('Prediction accuracy of model:', sum(acc)/len(acc))print('Training time for KNN:', time.time()-start)

In [ ]:
start=time.time()acc=[]for i in range(10):    X_train, X_test, y_train, y_test = train_test_split(X_std,y,test_size=0.2,random_state=i)    num_cores = multiprocessing.cpu_count()    inputs = range(len(X_test))    def predict_classification(train_X, train_y, test_row, k):        distances=[(euclidean(test_row, row), train_y[idx]) for idx,row in enumerate(train_X)]        distances.sort(key=lambda tup:tup[0])        neighbors=[label for _,label in distances[:k]]        return max(set(neighbors), key=neighbors.count)    def multi_run(t):        return predict_classification(X_train,y_train,X_test[t],25)    results = Parallel(n_jobs=num_cores)(delayed(multi_run)(i) for i in inputs)    acc.append(test_accuracy(y_test, results))print('Prediction accuracy of model:', sum(acc)/len(acc))print('Training time for KNN:', time.time()-start)